# Manually review the collection-dataset QA candidates

`collection_qa_flagged_candidates.csv` (built by a regex heuristic pass over
`final_collection_final.parquet` -- the deduped-against-OpenSLR, hidden-char-cleaned
collection dataset, 4,904 rows) lists 268 rows that matched one or more suspicious
patterns, mirroring the categories used for the OpenSLR SinSpeech review:

- **A_number_fragment** (49) -- bare `ක`/`ක්`/`කට`/`කින්` where a numeral might
  be missing. **Noisy**: `කට`/`ක` is also a normal Sinhala dative postposition
  ("to/for"), so most hits here are real Sinhala, not errors.
- **B_duplicated_word** (199) -- immediately repeated word. **Noisy**: Sinhala
  legitimately reduplicates words for emphasis/distributivity (`ඒ ඒ`, `එක එක`,
  `ලොකු ලොකු`), so most hits here are real Sinhala too.
- **C_garbled_conjunct** (1) -- vowel sign before the conjunct instead of after
  (e.g. `පේ්‍රම` instead of `ප්‍රේම`). Higher confidence -- looks like a real typo.
  (Most of the 13 originally found here were exact OpenSLR duplicates and got
  dropped by dedup -- only 1 survives in the final dataset.)
- **D_stray_period** (4) -- a lone `.` disconnected from any number/sentence
  boundary. Higher confidence -- looks like a real glitch.
- **E_initials** (22) -- period inside a token. Mixed: some are legitimate
  abbreviations (`එල්.ටී.ටී.ඊ` = LTTE), some are a missing-space-after-period bug
  (two sentences merged) -- a text fix, not necessarily a delete.

None of this was verified against audio like the OpenSLR list was -- it's text
pattern matching only. Listen to each clip yourself before deciding.

4 rows carried forward from an earlier review pass that (incorrectly) ran
against the pre-dedup 15,002-row file -- those already have logged decisions
and won't be re-queued.

**Workflow:** run the review-loop cell, listen to each clip, type a one-letter
decision. Every decision is written to `collection_qa_manual_review_log.csv`
immediately -- close the notebook anytime and rerun the loop cell later to pick
up where you left off (already-reviewed rows are skipped automatically).

In [ ]:
import datetime
import io
import os

import pandas as pd
import soundfile as sf
from IPython.display import Audio, clear_output, display

FINAL_DIR = "../../data/final_dataset"
FULL_PARQUET = os.path.join(FINAL_DIR, "final_collection_final.parquet")
OUT_PARQUET = os.path.join(FINAL_DIR, "final_collection_qa.parquet")

DATA_DIR = "../../../../data/processed"
CANDIDATES_CSV = os.path.join(DATA_DIR, "collection_qa_flagged_candidates.csv")
REVIEW_LOG = os.path.join(DATA_DIR, "collection_qa_manual_review_log.csv")

CATEGORY_COLS = ["A_number_fragment", "B_duplicated_word", "C_garbled_conjunct", "D_stray_period", "E_initials"]

## Load candidates + resume state

Safe to rerun this cell anytime -- it just re-reads whatever's on disk. `row_id`
is the row's positional index in `final_collection_final.parquet`, so it only
stays valid as long as that file isn't rewritten/reordered in the meantime.

In [ ]:
candidates = pd.read_csv(CANDIDATES_CSV, index_col=0)
candidates.index.name = "row_id"
candidates["category"] = candidates[CATEGORY_COLS].apply(
    lambda r: ",".join(c for c in CATEGORY_COLS if r[c]), axis=1
)

print(f"{len(candidates)} candidate rows to review")
print(candidates["category"].value_counts())

if os.path.exists(REVIEW_LOG):
    log = pd.read_csv(REVIEW_LOG, dtype=str, keep_default_na=False)
else:
    log = pd.DataFrame(columns=["row_id", "category", "original_text", "final_text", "action", "reviewed_at"])

reviewed_ids = set(log["row_id"].astype(int)) if len(log) else set()
remaining = candidates[~candidates.index.isin(reviewed_ids)]
print(f"\nalready reviewed: {len(reviewed_ids)} / {len(candidates)}")
print(f"remaining: {len(remaining)}")

268 candidate rows to review
category
B_duplicated_word                      192
A_number_fragment                       47
E_initials                              19
B_duplicated_word,E_initials             3
D_stray_period                           2
B_duplicated_word,D_stray_period         2
A_number_fragment,B_duplicated_word      2
C_garbled_conjunct                       1
Name: count, dtype: int64

already reviewed: 12 / 268
remaining: 256


## Optional: focus on one category at a time

Leave `CATEGORY_FILTER = None` to go through everything, or set it (e.g.
`"D_stray_period"`) to only queue that category this session. Given the noise
in A/B, you may want to only spot-check a handful of those rather than
reviewing all ~248 of them one by one.

In [ ]:
CATEGORY_FILTER = "B_duplicated_word,D_stray_period"  # e.g. "C_garbled_conjunct", "D_stray_period", "E_initials", ... or None for all

queue = candidates.loc[remaining.index]
if CATEGORY_FILTER is not None:
    queue = queue[queue["category"].str.contains(CATEGORY_FILTER, na=False)]
print(f"queued {len(queue)} rows" + (f" (category contains {CATEGORY_FILTER})" if CATEGORY_FILTER else ""))

queued 4 rows (category contains D_stray_period)


## Load audio

`final_collection_final.parquet` already stores ready-to-play WAV bytes (no flac
rebuild step needed, unlike OpenSLR). This reads the full `audio` column once so
the review loop below can look up any row by `row_id` without re-reading the
parquet on every clip.

In [ ]:
print("loading audio column (reads the full parquet once)...")
audio_lookup = pd.read_parquet(FULL_PARQUET, columns=["audio"])
print(f"loaded {len(audio_lookup)} rows")

loading audio column (reads the full parquet once)...


loaded 4904 rows


## Review loop

For each clip: listens, shows the transcript + which category(ies) matched, then
asks for a decision --

- `k` -- keep text exactly as-is (the flagged pattern is fine / not an error)
- `e` -- edit: type a corrected transcript
- `d` -- delete: confirm this row should be dropped from training data
- `s` -- skip for now, ask again next session
- `q` -- stop reviewing, save and exit (safe -- nothing is lost)

Saves to `collection_qa_manual_review_log.csv` after **every single decision**,
so an interrupted kernel never loses more than the row in progress.

In [8]:
def save_log():
    log.to_csv(REVIEW_LOG, index=False)


for row_id, row in queue.iterrows():
    clear_output(wait=True)
    print(f"row_id: {row_id}   category: {row['category']}   source: {row['source_dataset']}")
    print(f"text:   {row['text']!r}")

    audio_bytes = audio_lookup.loc[row_id, "audio"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    display(Audio(data=data, rate=sr, autoplay=True))

    action = input("[k]eep / [e]dit / [d]elete / [s]kip / [q]uit: ").strip().lower()

    if action == "q":
        print("stopped -- progress saved, rerun this cell to resume")
        break
    if action == "s":
        continue

    if action == "e":
        new_text = input(f"corrected text (was: {row['text']!r}): ").strip()
        final_text, act = new_text, "edited"
    elif action == "d":
        final_text, act = row["text"], "delete"
    else:
        final_text, act = row["text"], "keep"

    log.loc[len(log)] = [
        row_id, row["category"], row["text"], final_text, act,
        datetime.datetime.now().isoformat(timespec="seconds"),
    ]
    save_log()

print(f"\nsession done. total reviewed so far: {len(log)} / {len(candidates)}")

row_id: 1224   category: D_stray_period   source: youtube
text:   'මේකේ VFX නුත් මාර ඒවා නෑ .ඉන්න මිනිස්සුත් සාමාන්\u200dය මිනිස්සු අතර පොඩි නිකං comforting ගතියක් දැනුණා. මට එක බලද්දි'


KeyboardInterrupt: Interrupted by user

## Progress summary

In [ ]:
if len(log):
    print(log["action"].value_counts())
    print(f"\n{len(candidates) - len(log)} rows still unreviewed")
else:
    print("no rows reviewed yet")

action
keep      2
delete    1
edited    1
Name: count, dtype: int64

264 rows still unreviewed


## Apply reviewed decisions to a new parquet

**Not run automatically -- read before running.** Only acts on rows that have an
explicit human decision in `collection_qa_manual_review_log.csv`; anything not
yet reviewed (including every row that was never flagged at all) is left
untouched. Writes a **new** `final_collection_qa.parquet` -- does not overwrite
`final_collection_final.parquet`.

In [ ]:
APPLY_RESULTS = False  # flip to True when ready to fold collection_qa_manual_review_log.csv into a new parquet

if APPLY_RESULTS:
    full = pd.read_parquet(FULL_PARQUET)

    log_by_id = log.copy()
    log_by_id["row_id"] = log_by_id["row_id"].astype(int)
    log_by_id = log_by_id.set_index("row_id")
    delete_ids = set(log_by_id[log_by_id["action"] == "delete"].index)
    edit_map = log_by_id[log_by_id["action"] == "edited"]["final_text"].to_dict()

    before = len(full)
    full = full[~full.index.isin(delete_ids)]
    full["text"] = [edit_map.get(idx, t) for idx, t in zip(full.index, full["text"])]

    full = full.reset_index(drop=True)
    full.to_parquet(OUT_PARQUET, index=False)
    print(f"dropped {before - len(full)} rows (manually confirmed delete)")
    print(f"edited {len(edit_map)} rows")
    print(f"wrote {len(full)} rows -> {OUT_PARQUET}")
else:
    print("APPLY_RESULTS is False -- skipped. Set to True once you've reviewed what you want to apply.")